In [1]:
import cv2
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

In [2]:
nmf_video = Path("/Users/stimpfli/Desktop/prototype_data/poseforge/BO_Gal4_fly1_trial002/segment_029/nmf_sim_render_colorcode_0.mp4")
style_transfer_video_path = Path("/Users/stimpfli/Desktop/prototype_data/poseforge/test/epoch121_examplesim00.mp4")

In [12]:
# Output path for the overlay video
overlay_video_path = nmf_video.parent / f"{nmf_video.stem}_overlay_with_style.mp4"

cap_nmf = cv2.VideoCapture(str(nmf_video))
cap_style = cv2.VideoCapture(str(style_transfer_video_path))

if not cap_nmf.isOpened():
    raise RuntimeError(f"Could not open NMF video: {nmf_video}")
if not cap_style.isOpened():
    raise RuntimeError(f"Could not open style transfer video: {style_transfer_video_path}")

# Use NMF video properties as target
nmf_width = int(cap_nmf.get(cv2.CAP_PROP_FRAME_WIDTH))
nmf_height = int(cap_nmf.get(cv2.CAP_PROP_FRAME_HEIGHT))
crop_width = 400
crop_height = 400
crop_width_offset = (nmf_width - crop_width) // 2
crop_height_offset = (nmf_height - crop_height) // 2

nmf_fps = cap_nmf.get(cv2.CAP_PROP_FPS)
if not nmf_fps or np.isnan(nmf_fps) or nmf_fps <= 0:
    nmf_fps = 30.0

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(overlay_video_path), fourcc, nmf_fps, (3*crop_width, crop_height))

alpha = 0.2  # blend weight for NMF
beta = (1-alpha)   # blend weight for style transfer

frame_count = 0
while True:
    ret_nmf, frame_nmf = cap_nmf.read()
    ret_style, frame_style = cap_style.read()

    # Stop when either video ends
    if not ret_nmf or not ret_style:
        break

    # Resize style-transfer frame to NMF resolution
    frame_style_resized = cv2.resize(frame_style, (nmf_width, nmf_height), interpolation=cv2.INTER_LINEAR)

    # crop every frame to 400x400
    frame_nmf = frame_nmf[crop_height_offset:crop_height_offset+crop_height, crop_width_offset:crop_width_offset+crop_width]
    frame_style_resized = frame_style_resized[crop_height_offset:crop_height_offset+crop_height, crop_width_offset:crop_width_offset+crop_width]

    # Overlay frames
    overlay = cv2.addWeighted(frame_nmf, alpha, frame_style_resized, beta, 0)

    fin_frame = np.hstack((frame_nmf, frame_style_resized, overlay))

    writer.write(fin_frame)
    frame_count += 1

cap_nmf.release()
cap_style.release()
writer.release()

print(f"Overlay video written to: {overlay_video_path}")
print(f"Frames written: {frame_count}")

Overlay video written to: /Users/stimpfli/Desktop/prototype_data/poseforge/BO_Gal4_fly1_trial002/segment_029/nmf_sim_render_colorcode_0_overlay_with_style.mp4
Frames written: 1126
